In [ ]:
!pip install streamlit google-genai textblob feedparser beautifulsoup4 plotly pandas deep-translator git+https://github.com/GONZOsint/factcheckexplorer.git

In [ ]:
%%writefile app.py
import streamlit as st
import pytz
import os
from google import genai
from google.genai import types
from textblob import TextBlob
import feedparser
import pandas as pd
import plotly.express as px
import time
import json
import requests
from bs4 import BeautifulSoup
from datetime import datetime
from deep_translator import GoogleTranslator
from factcheckexplorer.factcheckexplorer import FactCheckLib


# --- 1. CONFIGURATION ---
st.set_page_config(
    page_title="SentinelNews Enterprise",
    page_icon="🛡️",
    layout="wide",
    initial_sidebar_state="expanded"
)

# --- 2. CSS STYLING ---
st.markdown("""
    <style>
    .main { background-color: #f8f9fa; font-family: 'Segoe UI', sans-serif; }
    h1, h2, h3 { color: #1e3a8a; font-weight: 700; }

    .news-card {
        background-color: white;
        color: #000000 !important;
        padding: 20px;
        border-radius: 12px;
        border-left: 5px solid #3b82f6;
        margin-bottom: 15px;
        box-shadow: 0 4px 6px rgba(0,0,0,0.05);
    }
    .news-card h4 { color: #1e3a8a !important; font-weight: 700; margin-bottom: 10px; }
    .news-card p { color: #333333 !important; font-size: 14px; margin-bottom: 10px; }
    .news-card a { color: #2563eb !important; font-weight: 600; text-decoration: none; }
    .news-card:hover { transform: translateY(-3px); box-shadow: 0 8px 15px rgba(0,0,0,0.1); }

    .fact-alert-box {
        background-color: #fff3cd !important;
        border-left: 6px solid #ffc107 !important;
        color: #856404 !important;
        padding: 20px;
        border-radius: 8px;
        margin-top: 15px;
        font-weight: 700;
        font-size: 16px;
    }

    .agent-log {
        font-family: 'Courier New', monospace;
        background-color: #1e1e1e;
        color: #00ff00;
        padding: 10px;
        border-radius: 5px;
        font-size: 12px;
        margin-bottom: 10px;
    }
    </style>
""", unsafe_allow_html=True)

if 'history' not in st.session_state: st.session_state.history = []
if 'gemini_client' not in st.session_state: st.session_state.gemini_client = None

# --- 3. MICROSERVICE CLASSES ---

class NewsAggregatorAgent:
    def fetch_headlines(self, source_name):
        sources = {
            "BBC World": "http://feeds.bbci.co.uk/news/world/rss.xml",
            "Times of India": "https://timesofindia.indiatimes.com/rssfeedstopstories.cms",
            "CNN Top Stories": "http://rss.cnn.com/rss/edition.rss"
        }
        if source_name not in sources: return []
        try:
            feed = feedparser.parse(sources[source_name])
            return [{
                "title": e.title,
                "link": e.link,
                "published": e.get('published', "Just Now"),
                "summary": e.get('summary', 'Summary unavailable.')[:300]
            } for e in feed.entries[:6]]
        except:
            return []

    def scrape_full_text(self, url):
        try:
            headers = {'User-Agent': 'Mozilla/5.0'}
            response = requests.get(url, headers=headers, timeout=5)
            soup = BeautifulSoup(response.content, 'html.parser')
            paragraphs = soup.find_all('p')
            full_text = ' '.join([p.get_text() for p in paragraphs])
            return full_text[:4000] if len(full_text) > 100 else None
        except:
            return None

class FactCheckerAgent:
    def __init__(self):

        pass

    def verify_claims(self, headline, language="en"):
        try:
            fact_check = FactCheckLib(query=headline, language=language)
            fact_check.process()

            if fact_check.results:
                top_match = fact_check.results[0]
                reviewer = top_match.get('reviewer', 'a verified journalist')
                rating = top_match.get('rating', 'Unknown')
                url = top_match.get('url', '#')
                return {
                    "status": "Warning",
                    "message": f"Fact-checked by {reviewer}. Rating: {rating}. Read more: {url}"
                }
            else:
                return {"status": "Verified", "message": "✅ No active fact-checks found in the database."}
        except Exception as e:
            return {"status": "Verified", "message": "Database check skipped or unavailable."}

class MultilingualExecutorAgent:
    def __init__(self, client):
        self.client = client
        self.translator = GoogleTranslator(source='auto', target='en')

    def analyze_and_rewrite(self, text):
        # 1. Translation Step
        try:
            english_text = self.translator.translate(text[:3000])
        except:
            english_text = text[:3000] # Fallback if offline

        # 2. LLM Step
        prompt = f"""Act as Executor Agent. Input text: "{english_text}".
        Tasks: 1. Detect bias. 2. Rewrite neutrally (AP Style).
        Return STRICT JSON exactly like this: {{
            "detected_language": "Language Name",
            "bias_score": 85,
            "bias_type": "Political or Emotional bias",
            "reasoning": "One sentence reasoning",
            "neutral_rewrite": "Rewritten text"
        }}"""

        try:
            res = self.client.models.generate_content(
                model='gemini-3.1-flash-lite',
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    safety_settings=[
                        types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH, threshold=types.HarmBlockThreshold.BLOCK_NONE),
                        types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_HARASSMENT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
                        types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT, threshold=types.HarmBlockThreshold.BLOCK_NONE)
                    ]
                )
            )
            result = json.loads(res.text.replace('```json','').replace('```','').strip())
            return result
        except Exception as e:
            # Prevents the "0" score bug by printing the actual error
            return {
                "detected_language": "Error",
                "bias_score": 0,
                "bias_type": f"API Error: {e}",
                "neutral_rewrite": f"🚨 THE API CRASHED. EXACT ERROR: {str(e)}"
            }

class PlannerAgent:
    def create_plan(self, input_type):
        return [
            "1. [Aggregator] Ingest & Extract Text",
            "2. [Translator] Python Deep-Translate to English",
            "3. [Fact-Checker] Query Journalist Database via API",
            "4. [Executor] LLM Bias Detection & Neutral Rewrite"
        ]

# --- 4. MAIN UI ---
with st.sidebar:
    st.image("https://cdn-icons-png.flaticon.com/512/9626/9626620.png", width=80)
    st.title("SentinelNews")
    st.caption("Major Project | Group 43")


    api_key = os.environ.get("GEMINI_API_KEY")

    if api_key:
        try:
            st.session_state.gemini_client = genai.Client(api_key=api_key)
            st.success("🟢 System Online ")
        except Exception as e:
            st.error(f"Failed to initialize AI: {e}")
    else:
        st.warning("⚠️ GEMINI_API_KEY environment variable not found. The app needs this to run AI models.")


    st.markdown("---")
    st.markdown("#### 🟢 Active Microservices")
    st.success("Web Scraper Engine")
    st.success("Deep-Translator Module")
    st.success("FactCheck DB Query")
    st.success("LLM Bias Executor")

st.title("SentinelNews Enterprise 🛡️")
st.markdown("**A Cross-Lingual, Autonomous News Auditing Framework**")

tab1, tab2, tab3 = st.tabs(["🔴 Real-Time Feed", "📝 Manual Audit", "📊 Analytics"])

# TAB 1: REAL-TIME FEED
with tab1:
    col1, col2 = st.columns([3, 1])
    with col1:
        src = st.selectbox("News Source", ["BBC World", "Times of India", "CNN Top Stories"])
    with col2:
        st.write("")
        if st.button("🔄 Refresh Feed"):
            with st.spinner("Aggregator scraping headlines..."):
                st.session_state.headlines = NewsAggregatorAgent().fetch_headlines(src)

    if 'headlines' in st.session_state:
        for idx, art in enumerate(st.session_state.headlines):
            st.markdown(f"""
            <div class="news-card">
                <h4>{art['title']}</h4>
                <p><strong>Published:</strong> {art['published']}</p>
                <p>{art['summary']}</p>
                <p><a href="{art['link']}" target="_blank">🔗 Click here to read full article</a></p>
            </div>
            """, unsafe_allow_html=True)

            if st.button(f"🛡️ Fetch Full Text & Audit #{idx+1}", key=f"btn_{idx}"):
                with st.spinner("Scraping full content..."):
                    aggregator = NewsAggregatorAgent()
                    full_content = aggregator.scrape_full_text(art['link'])

                    if full_content:
                        # Pass title first so Fact-Checker doesn't crash on long text
                        st.session_state.manual_input = f"{art['title']}\n\n{full_content}"
                        st.success("✅ Full article scraped successfully!")
                    else:
                        st.warning("Could not scrape full text. Using summary instead.")
                        st.session_state.manual_input = f"{art['title']}\n\n{art['summary']}"

                    st.info("Loaded into Manual Audit tab! Click 'Manual Audit' above.")

# TAB 2: MANUAL AUDIT
with tab2:
    txt = st.text_area("Input Text (Headline on first line, followed by article)", value=st.session_state.get('manual_input', ''), height=300)

    if st.button("🚀 Start Audit") and txt:
        if not st.session_state.gemini_client:
            st.error("⚠️ Please configure your Gemini API Key in the environment!")
        else:
            planner = PlannerAgent()
            executor = MultilingualExecutorAgent(st.session_state.gemini_client)
            checker = FactCheckerAgent()

            # Extract just the headline for the database check
            headline = txt.split('\n')[0]

            with st.status("🧠 Pipeline Initializing...", expanded=True) as status:
                for step in planner.create_plan("Manual"):
                    st.markdown(f"<div class='agent-log'>{step}</div>", unsafe_allow_html=True)
                    time.sleep(0.5)
                status.update(label="Planning Complete", state="complete", expanded=False)

            with st.spinner("🕵️ Fact-Checking DB & ✍️ Analyzing Bias..."):
                fact_res = checker.verify_claims(headline)
                exec_res = executor.analyze_and_rewrite(txt)

            m1, m2, m3 = st.columns(3)
            m1.metric("Bias Score", f"{exec_res.get('bias_score',0)}/100")
            m2.metric("Fact Status", fact_res.get('status','Unknown'))
            m3.metric("Language", exec_res.get('detected_language','En'))

            if fact_res.get('status') == "Warning":
                st.markdown(f"""<div class="fact-alert-box">{fact_res.get('message', 'Issue detected.')}</div>""", unsafe_allow_html=True)
            else:
                st.success("✅ Fact-Check Passed")

            c1, c2 = st.columns(2)
            with c1: st.info("Original"); st.write(txt[:1000] + "...")
            with c2: st.success("Neutral Rewrite"); st.write(exec_res.get('neutral_rewrite','Error during rewrite.'))

            # --- UPDATED TIMEZONE LOGIC ---
            ist_timezone = pytz.timezone('Asia/Kolkata')
            local_time = datetime.now(ist_timezone).strftime("%H:%M")

            st.session_state.history.append({
                "Time": local_time,
                "Score": exec_res.get('bias_score',0),
                "Type": exec_res.get('bias_type','N/A')
            })

# TAB 3: ANALYTICS
with tab3:
    if st.session_state.history:
        df = pd.DataFrame(st.session_state.history)
        st.plotly_chart(px.line(df, x="Time", y="Score", title="Bias Trend"), use_container_width=True)
        st.dataframe(df, use_container_width=True)
    else:
        st.info("No data yet. Run an audit to see analytics.")

In [ ]:
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
import subprocess
import time

subprocess.Popen(["streamlit", "run", "app.py"])
time.sleep(5)
!./cloudflared tunnel --url http://localhost:8501

In [ ]:
import time, json
from google import genai
from google.genai import types
from textblob import TextBlob
from deep_translator import GoogleTranslator
from factcheckexplorer.factcheckexplorer import FactCheckLib
from google.colab import userdata

# ── SECURE API KEY FETCHING ──────────────────────────────────────────────────

try:
    API_KEY = userdata.get('GEMINI_API_KEY')
    client  = genai.Client(api_key=API_KEY)
except userdata.SecretNotFoundError:
    print("❌ ERROR: 'GEMINI_API_KEY' not found in Colab Secrets.")
    print("Please click the key icon on the left, add a new secret named GEMINI_API_KEY with your key, and enable 'Notebook access'.")
    raise SystemExit()


# ── AGENT CODE WITH BUILT-IN COLD START FAULT TOLERANCE ──────────────────────
class FactCheckerAgent:
    def verify_claims(self, headline, language="en"):
        try:
            fc = FactCheckLib(query=headline, language=language)
            fc.process()
            if fc.results:
                top = fc.results[0]
                return {
                    "status":  "Warning",
                    "message": f"Fact-checked by {top.get('reviewer','journalist')}. "
                               f"Rating: {top.get('rating','Unknown')}."
                }
            return {"status": "Verified", "message": "No active fact-checks found."}
        except Exception as e:
            return {"status": "Verified", "message": f"DB check skipped: {e}"}

class MultilingualExecutorAgent:
    def __init__(self, client):
        self.client     = client
        self.translator = GoogleTranslator(source='auto', target='en')

    def analyze_and_rewrite(self, text):
        try:
            english_text = self.translator.translate(text[:3000])
        except:
            english_text = text[:3000]

        prompt = f"""Act as Executor Agent. Input text: "{english_text}".
Tasks: 1. Detect bias. 2. Rewrite neutrally (AP Style).
Return STRICT JSON exactly like this: {{
    "detected_language": "Language Name",
    "bias_score": 85,
    "bias_type": "Political or Emotional bias",
    "reasoning": "One sentence reasoning",
    "neutral_rewrite": "Rewritten text"
}}"""

        max_retries = 3
        for attempt in range(max_retries):
            try:
                res = self.client.models.generate_content(
                    model='gemini-3.1-flash-lite',
                    contents=prompt,
                    config=types.GenerateContentConfig(
                        response_mime_type="application/json",
                        safety_settings=[
                            types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,       threshold=types.HarmBlockThreshold.BLOCK_NONE),
                            types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_HARASSMENT,        threshold=types.HarmBlockThreshold.BLOCK_NONE),
                            types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
                        ]
                    )
                )
                return json.loads(res.text.replace('```json','').replace('```','').strip())

            except Exception as e:
                # Catch transient 503 connection spikes and retry after 3 seconds
                if "503" in str(e) and attempt < max_retries - 1:
                    print(f"   ⚠️ 503 Server Busy on attempt {attempt + 1}. Retrying connection in 3 seconds...")
                    time.sleep(3)
                    continue
                else:
                    return {"detected_language":"Error","bias_score":0,
                            "bias_type":f"API Error: {e}","neutral_rewrite":str(e)}

# ── 8 BENCHMARK TEST CASES ───────────────────────────────────────────────────
TEST_CASES = [
    {
        "id": 1, "language": "English", "ground_truth": True,
        "description": "Vaccine microchip conspiracy",
        "headline": "COVID vaccines contain microchips used by Bill Gates to track the population",
        "text": ("COVID-19 vaccines have been secretly loaded with microscopic tracking chips "
                 "developed by Bill Gates and the globalist elite. The chips connect to 5G towers "
                 "to monitor your every movement and thought. Thousands of whistleblowers confirm "
                 "the government is hiding this truth from the public. The mainstream media refuses "
                 "to report on this because they are part of the conspiracy.")
    },
    {
        "id": 2, "language": "English", "ground_truth": True,
        "description": "Temporal election claim (ambiguous)",
        "headline": "Party X has officially won the election with a landslide majority",
        "text": ("Party X has officially won the election with a landslide majority, "
                 "securing over 400 seats in parliament. The opposition has been completely "
                 "wiped out and their leadership has resigned in disgrace. The people have "
                 "spoken and the corrupt establishment has finally been defeated forever.")
    },
    {
        "id": 3, "language": "English", "ground_truth": False,
        "description": "Chandrayaan-3 lunar mission (factual)",
        "headline": "India's Chandrayaan-3 successfully lands near Moon's south pole",
        "text": ("India's Chandrayaan-3 spacecraft successfully landed near the Moon's south pole "
                 "on August 23, 2023, making India the fourth country to achieve a soft lunar "
                 "landing and the first to land near the lunar south pole. The Vikram lander "
                 "and Pragyan rover were deployed as planned. ISRO scientists confirmed all "
                 "systems are functioning normally.")
    },
    {
        "id": 4, "language": "English", "ground_truth": False,
        "description": "Sarcastic political commentary",
        "headline": "Oh great, another brilliant policy from our wonderful leaders",
        "text": ("Oh great, another absolutely brilliant policy decision from our wonderfully "
                 "competent elected representatives. Because nothing says good governance like "
                 "announcing a new tax on rain while simultaneously cutting healthcare budgets. "
                 "Our leaders have once again demonstrated their remarkable genius and their "
                 "deep concern for the ordinary citizen. Truly inspiring stuff.")
    },
    {
        "id": 5, "language": "English", "ground_truth": False,
        "description": "Ideological opinion piece",
        "headline": "Free market capitalism is destroying the middle class and must be replaced",
        "text": ("Free market capitalism has systematically destroyed the middle class over the "
                 "past four decades, transferring wealth upward to a tiny plutocratic elite. "
                 "Workers are exploited, wages stagnate, and corporations pay no taxes while "
                 "ordinary families struggle to survive. Only by dismantling this corrupt system "
                 "and replacing it with socialist policies can we restore justice and dignity.")
    },
    {
        "id": 6, "language": "English", "ground_truth": True,
        "description": "Flat earth theory",
        "headline": "NASA admits the Earth is actually flat and has been hiding this for decades",
        "text": ("NASA scientists have finally been forced to admit that the Earth is flat, "
                 "not a globe. Decades of satellite imagery have been computer-generated "
                 "fabrications designed to maintain the spherical Earth hoax. The flat Earth "
                 "is covered by a dome called the firmament, and Antarctica is actually a "
                 "massive ice wall surrounding the edge. Pilots, sailors and astronauts are "
                 "all sworn to secrecy about the true shape of our world.")
    },
    {
        "id": 7, "language": "Hindi", "ground_truth": False,
        "description": "Hindi factual news",
        "headline": "भारत ने मंगल मिशन के लिए नया अंतरिक्ष यान लॉन्च किया",
        "text": ("भारतीय अंतरिक्ष अनुसंधान संगठन (इसरो) ने मंगल ग्रह के अध्ययन के लिए "
                 "एक नया अंतरिक्ष यान सफलतापूर्वक लॉन्च किया है। यह मिशन मंगल की सतह "
                 "और वायुमंडल का विस्तृत अध्ययन करेगा। वैज्ञानिकों ने बताया कि यान अगले "
                 "सात महीनों में मंगल की कक्षा में पहुंचेगा और डेटा संग्रह शुरू करेगा।")
    },
    {
        "id": 8, "language": "Hindi", "ground_truth": True,
        "description": "Hindi disinformation",
        "headline": "सरकार ने गुप्त रूप से पीने के पानी में दवाइयां मिलाई हैं",
        "text": ("सरकार ने बिना किसी सूचना के नागरिकों के पीने के पानी में गुप्त दवाइयां "
                 "मिला दी हैं जो लोगों की सोचने की क्षमता को कम करती हैं। यह एक बड़ी "
                 "साजिश है जिसे मुख्यधारा मीडिया छुपा रही है। हजारों लोग इस जहर के "
                 "कारण बीमार पड़ रहे हैं लेकिन सरकार सच्चाई दबा रही है।")
    },
]

# ── RUN EVALUATION ───────────────────────────────────────────────────────────
checker  = FactCheckerAgent()
executor = MultilingualExecutorAgent(client)

results = []
print("=" * 80)
print("SENTINELNEWS LIVE EVALUATION RUN")
print("=" * 80)

for tc in TEST_CASES:
    print(f"\n>>> CASE {tc['id']} | {tc['language']} | {tc['description']}")
    print(f"    Headline: {tc['headline'][:70]}...")

    t_start = time.time()

    # Step 1: Fact-check using FactCheckLib
    lang_code = "hi" if tc["language"] == "Hindi" else "en"
    fact_res  = checker.verify_claims(tc["headline"], language=lang_code)

    # Step 2: Bias analysis using deep_translator + Gemini 3.1 Flash-Lite
    exec_res = executor.analyze_and_rewrite(tc["text"])

    t_end   = time.time()
    latency = round(t_end - t_start, 2)

    bias_score    = exec_res.get("bias_score", 0)
    detected_lang = exec_res.get("detected_language", "Unknown")
    fact_status   = fact_res.get("status", "Unknown")
    bias_type     = exec_res.get("bias_type", "").lower()

    # Dynamic mapping logic for Evaluation: Flagged as fake if structural database check
    # catches it or if semantic analysis scores heavy bias under a deceptive framework type.
    if fact_status == "Warning" or (bias_score > 80 and ("conspiracy" in bias_type or "misinformation" in bias_type or "political" in bias_type)):
        predicted_fake = True
    else:
        predicted_fake = False

    print(f"    Detected Language : {detected_lang}")
    print(f"    Bias Score        : {bias_score}/100")
    print(f"    Bias Type         : {exec_res.get('bias_type','N/A')}")
    print(f"    Fact Status       : {fact_status}")
    print(f"    Latency           : {latency}s")
    print(f"    Ground Truth Fake : {tc['ground_truth']}  |  Predicted Fake: {predicted_fake}")
    print(f"    Reasoning         : {exec_res.get('reasoning','')[:100]}")

    results.append({
        "id":             tc["id"],
        "language":       tc["language"],
        "description":    tc["description"],
        "ground_truth":   tc["ground_truth"],
        "bias_score":     bias_score,
        "fact_status":    fact_status,
        "predicted_fake": predicted_fake,
        "detected_lang":  detected_lang,
        "latency":        latency,
        "reasoning":      exec_res.get("reasoning",""),
        "neutral_rewrite":exec_res.get("neutral_rewrite",""),
    })


    print(f"⏳ Respecting API Rate Limits: Cooldown sleep for 12 seconds...")
    time.sleep(12)

# ── COMPUTE METRICS ──────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("METRIC COMPUTATION")
print("=" * 80)

TP = sum(1 for r in results if     r["ground_truth"] and     r["predicted_fake"])
TN = sum(1 for r in results if not r["ground_truth"] and not r["predicted_fake"])
FP = sum(1 for r in results if not r["ground_truth"] and     r["predicted_fake"])
FN = sum(1 for r in results if     r["ground_truth"] and not r["predicted_fake"])

precision   = TP / (TP + FP) if (TP + FP) > 0 else 0
recall      = TP / (TP + FN) if (TP + FN) > 0 else 0
f1          = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
accuracy    = (TP + TN) / len(results)
sensitivity = recall
mean_lat    = round(sum(r["latency"] for r in results) / len(results), 2)

hindi_res  = [r for r in results if r["language"] == "Hindi"]
hindi_acc  = sum(1 for r in hindi_res if r["ground_truth"] == r["predicted_fake"]) / len(hindi_res) if hindi_res else 0

print(f"\nConfusion Matrix:")
print(f"  TP={TP}  FP={FP}")
print(f"  FN={FN}  TN={TN}")
print(f"\nAggregated Metrics:")
print(f"  F1-Score             : {f1:.4f}  → rounded: {round(f1,2)}")
print(f"  Precision            : {precision:.4f}  → rounded: {round(precision,2)}")
print(f"  Recall               : {recall:.4f}  → rounded: {round(recall,2)}")
print(f"  Classification Acc.  : {accuracy*100:.1f}%  ({TP+TN}/{len(results)} correct)")
print(f"  Fact-Check Sensitivity: {sensitivity*100:.1f}%")
print(f"  Mean Latency         : {mean_lat}s")
print(f"  Hindi Accuracy       : {hindi_acc*100:.0f}%")

print("\n" + "=" * 80)
print("PER-CASE TABLE (copy into paper)")
print("=" * 80)
print(f"{'ID':<4} {'Lang':<8} {'Fake?':<6} {'Predicted':<10} {'Score':<6} {'Latency':<10} {'Match?'}")
print("-" * 60)
for r in results:
    pred_str = "Warning" if r["predicted_fake"] else "Verified"
    match    = "✓" if r["ground_truth"] == r["predicted_fake"] else "✗ WRONG"
    print(f"{r['id']:<4} {r['language']:<8} {str(r['ground_truth']):<6} {pred_str:<10} "
          f"{r['bias_score']:<6} {r['latency']:<10} {match}")

print("\n" + "=" * 80)
print("FULL JSON (save this for reference)")
print("=" * 80)
print(json.dumps(results, indent=2, ensure_ascii=False))